In [5]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    instance='crn:v1:bluemix:public:quantum-computing:us-east:a/1e275a2206db45a08b19f2545e2ae8f5:46562652-ebad-4e2b-a27a-1531217eb801::'
)
job = service.job('dbm9kpu74qds73drsr20')
job_result = job.result()

In [6]:
print("Job status:", job.status())
print("Backend:", job.backend().name)
print("Number of PUBs:", len(job_result))

for i, pub in enumerate(job_result):
    print(f"\n--- PUB {i} ---")
    print("Metadata:", pub.metadata)
    print("Available registers:", pub.data.keys())

    for register_name in pub.data.keys():
        register = getattr(pub.data, register_name)
        print(f"\nRegister: {register_name}")
        print("Counts:", register.get_counts())

Job status: DONE
Backend: ibm_fez
Number of PUBs: 1

--- PUB 0 ---
Metadata: {'circuit_metadata': {}, 'num_randomizations': 32, 'compilation': {'scheduler_timing': None, 'stretch_values': None}}
Available registers: dict_keys(['meas'])

Register: meas
Counts: {'00001': 128, '11100': 151, '01010': 129, '11010': 122, '01011': 120, '00110': 122, '01001': 147, '00000': 124, '11000': 123, '00011': 117, '10101': 98, '00010': 119, '11011': 111, '01111': 114, '01101': 164, '11110': 108, '10111': 112, '10010': 99, '00111': 99, '01000': 170, '01100': 312, '00100': 140, '10110': 85, '10001': 106, '11111': 115, '10011': 124, '11101': 125, '10100': 131, '00101': 130, '11001': 94, '10000': 106, '01110': 151}


In [7]:
import pandas as pd

all_data = []

for pub_index, pub in enumerate(job_result):

    counts = pub.data.meas.get_counts()

    total_shots = sum(counts.values())

    for state, count in counts.items():
        all_data.append({
            "PUB": pub_index,
            "State": state,
            "Counts": count,
            "Probability": count / total_shots,
            "Probability (%)": count / total_shots * 100
        })

df = pd.DataFrame(all_data)

print(df)

    PUB  State  Counts  Probability  Probability (%)
0     0  00001     128     0.031250         3.125000
1     0  11100     151     0.036865         3.686523
2     0  01010     129     0.031494         3.149414
3     0  11010     122     0.029785         2.978516
4     0  01011     120     0.029297         2.929688
5     0  00110     122     0.029785         2.978516
6     0  01001     147     0.035889         3.588867
7     0  00000     124     0.030273         3.027344
8     0  11000     123     0.030029         3.002930
9     0  00011     117     0.028564         2.856445
10    0  10101      98     0.023926         2.392578
11    0  00010     119     0.029053         2.905273
12    0  11011     111     0.027100         2.709961
13    0  01111     114     0.027832         2.783203
14    0  01101     164     0.040039         4.003906
15    0  11110     108     0.026367         2.636719
16    0  10111     112     0.027344         2.734375
17    0  10010      99     0.024170         2.

In [8]:
TARGET = "01100"

summary = []

for pub_index, pub in enumerate(job_result):

    counts = pub.data.meas.get_counts()

    shots = sum(counts.values())
    target_counts = counts.get(TARGET, 0)

    probability = target_counts / shots

    summary.append({
        "PUB": pub_index,
        "Shots": shots,
        "Target Counts": target_counts,
        "Success Probability": probability,
        "Success (%)": probability * 100
    })

summary_df = pd.DataFrame(summary)

print(summary_df)

   PUB  Shots  Target Counts  Success Probability  Success (%)
0    0   4096            312             0.076172     7.617188


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# AGGREGATE RESULTS
# ==========================================

results = []

for experiment, group in summary_df.groupby("Experiment"):

    shots = group["Shots"].sum()
    target_counts = group["Target Counts"].sum()

    probability = target_counts / shots

    stderr = np.sqrt(
        probability * (1 - probability) / shots
    )

    results.append({
        "Experiment": experiment,
        "Shots": shots,
        "Success (%)": probability * 100,
        "Standard Error (%)": stderr * 100
    })

analysis_df = pd.DataFrame(results)

# ==========================================
# EXCLUDE HIGHEST-PERFORMING CONFIGURATION
# ==========================================

highest_index = analysis_df["Success (%)"].idxmax()

comparison_df = analysis_df.drop(
    index=highest_index
)

comparison_average = comparison_df["Success (%)"].mean()

analysis_df["Average of Other Configurations (%)"] = [
    analysis_df.loc[
        analysis_df.index != idx, "Success (%)"
    ].mean()
    for idx in analysis_df.index
]

analysis_df["Difference from Others' Average (%)"] = (
    analysis_df["Success (%)"] -
    analysis_df["Average of Other Configurations (%)"]
)

print("\n===== RESULTS =====")
print(analysis_df.to_string(index=False))

print(
    f"\nHighest configuration: "
    f"{analysis_df.loc[highest_index, 'Experiment']}"
)

print(
    f"Average of remaining configurations: "
    f"{comparison_average:.3f}%"
)

analysis_df.to_csv(
    "grover_comparison_analysis.csv",
    index=False
)

# ==========================================
# VISUALIZATION
# ==========================================

plot_df = analysis_df.sort_values("Success (%)")

plt.figure(figsize=(12, 7))

plt.bar(
    plot_df["Experiment"],
    plot_df["Success (%)"],
    yerr=plot_df["Standard Error (%)"],
    capsize=4,
    label="Experimental success"
)

plt.axhline(
    comparison_average,
    linestyle="--",
    label=f"Average excluding highest ({comparison_average:.2f}%)"
)

plt.axhline(
    3.125,
    linestyle=":",
    label="Random guessing (3.125%)"
)

plt.ylabel("Probability of measuring 01100 (%)")
plt.xlabel("Mitigation configuration")
plt.title("Grover: Error Mitigation Comparison")

plt.xticks(rotation=35, ha="right")
plt.legend()
plt.tight_layout()

plt.savefig("grover_comparison.png", dpi=300)
plt.show()
